# Andmehõive — Labor 2: mõõteandmete kontroll ja analüüs

See analüüs kasutab olemasolevaid faile **nende päritolu arvestades**. 
- `data/pressure_20261025_legacy.csv` sisaldab 145 ehtsat pingelugemit, kuid **ei sisalda aja-, pumbaoleku ega usaldusväärset `raw/opamp` veergu**.
- Esimene osa kuvab ainult salvestatud **pinget mõõterea järjekorranumbri järgi**. See EI OLE ajateljega graafik.
- SNR ja spektrianalüüs käivituvad **ainult uue, vajalike veergudega ja kontrollitud proovivõtuajaga** CSV põhjal.
- Nominaalsed ADC-arvutused ei ole mõõdetud müratulemus.

Käivita Jupyter kaustas `Data-Aq/lab2`. Vajalikud: `pandas numpy matplotlib scipy`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data" / "pressure_20261025_legacy.csv").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data"
legacy = pd.read_csv(DATA / "pressure_20261025_legacy.csv")
assert len(legacy) == 145, "Recovered log must preserve 145 records."
assert legacy["config"].eq("unverified").all()
assert legacy["t_ms"].isna().all(), "Legacy log cannot have timestamps."
assert legacy["pump_state"].isna().all(), "Legacy log cannot have pump states."
display(legacy.head())
print("Vastuvõetud mõõteread:", len(legacy))
print("Pinge: %.3f–%.3f V" % (legacy.voltage_mv.min()/1000, legacy.voltage_mv.max()/1000))
print("NB! See fail ei sobi SNR-i ega 100 Hz spektri arvutamiseks.")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(legacy["sample_index"], legacy["voltage_mv"] / 1000, lw=1)
ax.set(xlabel="Mõõterea järjekorranumber (mitte aeg!)",
       ylabel="Logitud Voltage (V)",
       title="25.10.2026 pinge muutumine — ajatempliteta alglogi")
ax.grid(alpha=.3)
plt.show()

## Teoreetiline ADC samm

Arvutuse eeldused: MPX5700AP anduritoide 5,06 V, nominaalne ülekandetundlikkus 0,0012858 1/kPa, 12-bitine 0–3,3 V ADC ja parandatud astme 1,37× võimendus. **Ei ole mõõdetud ADC tegelik täpsus või SNR.**

In [ ]:
adc_lsb_v = 3.3 / 4096
sensitivity_v_per_kpa = 5.06 * 0.0012858
raw_pa_per_step = 1000 * adc_lsb_v / sensitivity_v_per_kpa
opamp_pa_per_step = raw_pa_per_step / 1.37
print(f"Toorsignaal: {raw_pa_per_step:.1f} Pa / ADC samm")
print(f"Parandatud aste (teooria): {opamp_pa_per_step:.1f} Pa / ADC samm")

## Edasiste pärismõõtmiste kontroll: raw/opamp võrdlus

Vali uus võrdlus-CSV, kui see on päriselt salvestatud. Nõutud veerud:
`t_ms, config, trial_id, phase, voltage_mv`. Iga katse jaoks peab `t_ms` olema **mõõtmise kellaaeg millisekundites, mitte arvutisse saabumise aeg**. `config` on ainult `raw` või `opamp`; `phase` peab näitama vähemalt `baseline` ja `plateau`. Spektri ja SNR tulemusi ei koostata tehislikult tuletatud ajateljelt.

Järgmine plokk ei tee midagi, kui tõestatud võrdlusfaili ei ole. 

In [ ]:
comparison_path = DATA / "pressure_comparison_verified.csv"
if not comparison_path.exists():
    print("Võrdlusfail puudub — spektrit/SNR-i ei arvutata.")
    comparison = None
else:
    comparison = pd.read_csv(comparison_path)
    required = {"t_ms", "config", "trial_id", "phase", "voltage_mv"}
    missing = required - set(comparison.columns)
    if missing:
        raise ValueError(f"Võrdlusfailist puuduvad veerud: {sorted(missing)}")
    if comparison[list(required)].isna().any().any():
        raise ValueError("Võrdlusfailis on nõutud välju ilma väärtuseta.")
    if not set(comparison["config"].unique()) <= {"raw", "opamp"}:
        raise ValueError("Võrdlusrežiim peab olema raw või opamp.")
    if set(comparison["config"].unique()) != {"raw", "opamp"}:
        raise ValueError("Mõlemat konfiguratsiooni raw ja opamp on vaja.")
    for config, group in comparison.groupby("config"):
        trials = group["trial_id"].nunique()
        if trials < 30:
            raise ValueError(f"{config}: ainult {trials} katset; vaja vähemalt 30.")
        for trial_id, trial in group.groupby("trial_id"):
            times = np.sort(trial["t_ms"].to_numpy(dtype=float))
            diffs = np.diff(times)
            if len(diffs) < 10 or not np.all(diffs > 0):
                raise ValueError(f"{config}/{trial_id}: liiga vähe ajatempliga mõõtmisi.")
            fs = 1000 / np.median(diffs)
            if not (80 <= fs <= 120):
                raise ValueError(f"{config}/{trial_id}: hinnanguline fs={fs:.1f} Hz; ei vasta ligikaudu 100 Hz nõudele.")
    print("Vorming ja ajasamm sobivad; kontrolli veel füüsilist seadistust ja katsefaase.")
    display(comparison.groupby("config")["trial_id"].nunique().rename("impulsside_arv"))

In [ ]:
if comparison is not None:
    from scipy.signal import welch
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for config, group in comparison.groupby("config"):
        # Use one actual trial per configuration; never concatenate time-separated trials as one series.
        trial = next(iter(group.groupby("trial_id")))[1].sort_values("t_ms")
        t = trial["t_ms"].to_numpy(dtype=float)
        v = trial["voltage_mv"].to_numpy(dtype=float) / 1000
        axes[0].plot((t-t[0]) / 1000, v, label=f"{config} / üks katse")
        if len(v) >= 32:
            fs = 1000 / np.median(np.diff(t))
            f, power = welch(v - np.mean(v), fs=fs, nperseg=min(256, len(v)))
            axes[1].semilogy(f, power, label=config)
    axes[0].set(xlabel="Aeg katse algusest (s)", ylabel="Pinge (V)", title="Tegelikud ajateljed")
    axes[1].set(xlabel="Sagedus (Hz)", ylabel="PSD (V²/Hz)", title="Welch PSD; võrreldavad katsed")
    for ax in axes:
        ax.grid(alpha=.3)
        ax.legend()
    plt.show()

    # SNR is based on plateau minus baseline and plateau standard deviation.
    # This is VOLTAGE-domain SNR: to compare pressure-domain kPa, calibrate both
    # configurations to the same physical units first.
    result = []
    for config, group in comparison.groupby("config"):
        for trial_id, trial in group.groupby("trial_id"):
            baseline = trial.loc[trial.phase == "baseline", "voltage_mv"].to_numpy(dtype=float)
            plateau = trial.loc[trial.phase == "plateau", "voltage_mv"].to_numpy(dtype=float)
            if len(baseline) < 5 or len(plateau) < 5:
                continue
            signal = abs(np.mean(plateau) - np.mean(baseline))
            noise = np.std(plateau, ddof=1)
            if noise > 0 and signal > 0:
                result.append({"config": config, "trial_id": trial_id,
                               "snr_db_voltage": 20 * np.log10(signal/noise)})
    result = pd.DataFrame(result)
    if result.empty:
        print("SNR võrdlus puudub: baseline/plateau mõõtepunkte pole piisavalt.")
    else:
        display(result.groupby("config")["snr_db_voltage"].agg(["count", "mean", "median"]))
        print("NB! See on pingepõhine SNR. Õppejõu nõutud kPa-põhine arvutus vajab iga konfiguratsiooni kalibratsiooni.")